In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops,hog
from skimage.filters import sobel
import pandas as pd
import os




In [ ]:
from skimage.feature import graycomatrix, graycoprops, hog
from skimage.filters import sobel
# import mahotas

# Define directories
img_dir = "brain_tumors"  # Path where grayscale images are stored
csv_output = "csv/features.csv"
subsets = ["Training", "Testing"]
categories = ["notumor", "glioma", "meningioma", "pituitary"]
img_size = (64, 64)  # Ensure uniform image size

# Define GLCM properties to extract
glcm_features = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']

# Define label mapping
labeling = {
    'notumor': 0,
    'glioma': 1,
    'meningioma': 2,
    'pituitary': 3
}

def extract_glcm_features(img):
    # extraxt the glcm features
    glcm = graycomatrix(img, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    glcm_features = [graycoprops(glcm, prop)[0, 0] for prop in glcm_features]
    return glcm_features

def extract_edge_features(img):
 
    # extract the edge features using sobel and canny edge detection method
    sobel_edges = sobel(img).mean()  
    canny_edges = cv2.Canny(img, 100, 200)
    edge_density = np.sum(canny_edges) / (img_size[0] * img_size[1])  
    return [sobel_edges, edge_density]

def extract_shape_features(img):
   
    _, binary_img = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return [0] * 7  

    largest_contour = max(contours, key=cv2.contourArea)
    hu_moments = cv2.HuMoments(cv2.moments(largest_contour)).flatten()
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)  
    return hu_moments.tolist()

def extract_histogram_features(img):
    hist = cv2.calcHist([img], [0], None, [16], [0, 256])  # 16-bin histogram
    hist = cv2.normalize(hist, hist).flatten()  # Normalize histogram
    return hist.tolist()

def extract_hog_features(img):
    hog_features = hog(img, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True)
    return hog_features.tolist()

def extract_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Read grayscale image
    if img is None:
        return None  # Skip if the image cannot be read

    img = cv2.resize(img, img_size)  # Resize for consistency

    glcm = extract_glcm_features(img)
    edge = extract_edge_features(img)
    shape = extract_shape_features(img)
    histogram= extract_histogram_features(img)
    hog = extract_hog_features(img)

    return glcm + edge + shape + histogram + hog 

def feature_extraction():
    data = []
    labels = []
    dataset_type = []  
    for subset in subsets:  
        input_subset_path = os.path.join(img_dir, subset)

        for category in categories:
            input_folder = os.path.join(input_subset_path, category)
            label = labeling[category]

            if not os.path.exists(input_folder):
                print(f"Warning: Skipping missing category folder {input_folder}")
                continue

            for filename in os.listdir(input_folder):
                image_path = os.path.join(input_folder, filename)
                features = extract_features(image_path)

                if features is not None:
                    data.append(features)
                    labels.append(label)
                    dataset_type.append(subset) 

    
    df = pd.DataFrame(data)
    df["label"] = labels
    df["dataset"] = dataset_type  

  
    df.to_csv(csv_output, index=False)
    print(f"Feature has been saved at CSV file: {csv_output}")


feature_extraction()

Feature extraction completed. CSV saved at: csv/features.csv


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
import numpy as np
from sklearn.preprocessing import StandardScaler


df=pd.read_csv("csv/features.csv")
# Load the Iris dataset as an example
X = df.drop(columns=["label", "dataset"]).values  # Drop label and dataset columns
y = df["label"].values
dataset_type = df["dataset"].values  # Retrieve dataset type

# Step 1: Use the actual Training and Testing split
X_train = X[dataset_type == "Training"]
y_train = y[dataset_type == "Training"]
X_test = X[dataset_type == "Testing"]
y_test = y[dataset_type == "Testing"]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define the model with 7 layers



C:\Users\YASAS\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
